# Popularity Non-ML Baseline

In [1]:
# optional if you have trouble running the environments
import sys
sys.path.insert(0, "../src")

# regular imports
import pandas as pd
from remy.paths import PROJECT_ROOT
DATA = PROJECT_ROOT / "data" / "raw"

## Load data

In [8]:
recipes = pd.read_parquet(DATA / "recipes.parquet", columns=["id", "name"])
interactions = pd.read_parquet(
    DATA / "interactions.parquet", columns=["user_id", "recipe_id", "date", "rating"]
)
print("Recipes:", recipes.shape, "Interactions:", interactions.shape, "Unique users:", interactions["user_id"].nunique())

Recipes: (231637, 2) Interactions: (1132367, 4) Unique users: 226570


## Train/val/test split

Split by date (temporally) in a 80/10/10 breakdown. This is deterministic, so we can be sure of the same splits every time. 

In [3]:
t_val, t_test = interactions["date"].quantile([0.8, 0.9])
train = interactions[interactions["date"] < t_val]
val = interactions[(interactions["date"] >= t_val) & (interactions["date"] < t_test)]
test = interactions[interactions["date"] >= t_test]

print(f"train: {len(train):,} | val: {len(val):,} | test: {len(test):,}")
print(f"val from {t_val.date()}, test from {t_test.date()}")

train: 905,786 | val: 113,312 | test: 113,269
val from 2011-12-27, test from 2014-02-25


## Top 10 most popular recipes

We suggest the 10 most popular recipes seen in the train dataset, where popular just means it had a lot of interactions - whether 0, 1, or 5 stars. 

In [7]:
top10_ids = train["recipe_id"].value_counts().head(10).index
top10_set = set(top10_ids)
top10 = recipes.set_index("id").loc[top10_ids, "name"].to_frame().assign(interactions=train["recipe_id"].value_counts().head(10).to_numpy())
top10

,name,interactions
recipe_id,,
27208,to die for crock pot roast,1323
89204,crock pot chicken with black beans cream cheese,1301
39087,creamy cajun chicken pasta,1095
32204,whatever floats your boat brownies,1047
22782,jo mama s world famous spaghetti,928
67256,best ever banana cake with cream cheese frosting,909
54257,yes virginia there is a great meatloaf,839
69173,kittencal s italian melt in your mouth meatballs,775
68955,japanese mum s chicken,773


## Evaluation

**Hit Rate@10** and **Recall@10**, each computed for two definitions of "positive": rating ≥ 4, and rating = 5 (a harder bar). Ratings of 0 (no star given) are excluded for evaluation only. 

- **Hit Rate@10**: did any of a user's positive-rated eval recipes land in our top 10, averaged over every user with at least one rated eval non-zero interaction
- **Recall@10**: what fraction of a user's positive-rated test recipes landed in our top 10, averaged over every user with at least one rated eval non-zero interaction

In [5]:
eval_rated = val[val["rating"] > 0]
users_with_signal = eval_rated["user_id"].unique()

def evaluate(min_rating):
    positive = eval_rated[eval_rated["rating"] >= min_rating]
    positive_items_by_user = positive.groupby("user_id")["recipe_id"].agg(set)
    hits_by_user = positive_items_by_user.apply(lambda items: len(items & top10_set))

    hit_rate = (hits_by_user > 0).reindex(users_with_signal, fill_value=False).mean()
    recall = (hits_by_user / positive_items_by_user.apply(len)).mean()

    return hit_rate, recall

In [6]:
rows = []
for min_rating, label in [(4, "rating >= 4"), (5, "rating = 5")]:
    hit_rate, recall = evaluate(min_rating)
    rows.append({"positive threshold": label, "Hit Rate@10": hit_rate, "Recall@10": recall})

pd.DataFrame(rows)

,positive threshold,Hit Rate@10,Recall@10
0,rating >= 4,0.030711,0.020096
1,rating = 5,0.027106,0.020939
